# BBC News Topic Modeling with LDA

**Lab Objectives:**
1. Clean and pre-process the BBC News dataset appropriately.
2. Apply LDA topic modeling using the Gensim library.
3. Experiment with at least three different numbers of topics.
4. Visualize the model output with PyLDAvis and interpret the results.


## Part 1: Load BBC News Dataset

We load the BBC dataset articles from the 5 category folders (`business`, `entertainment`, `politics`, `sport`, `tech`) into a pandas DataFrame.


In [1]:
from pathlib import Path
import pandas as pd

bbc_dir = Path("data/bbc")
records = []

for cat_dir in sorted(bbc_dir.iterdir()):
    if cat_dir.is_dir():
        category = cat_dir.name
        for f in sorted(cat_dir.glob("*.txt")):
            text = f.read_text(encoding="latin-1").strip()
            records.append({
                "category": category,
                "filename": f.name,
                "text": text
            })

df = pd.DataFrame(records)
print(f"Total documents loaded: {len(df)}")
print("\nArticles per category:")
print(df["category"].value_counts())
df.head(3)


Total documents loaded: 2225

Articles per category:
category
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64


,category,filename,text
0,business,001.txt,Ad sales boost Time Warner profit\n\nQuarterly...
1,business,002.txt,Dollar gains on Greenspan speech\n\nThe dollar...
2,business,003.txt,Yukos unit buyer faces loan claim\n\nThe owner...


## Part 2: Clean and Pre-process the Data

Our preprocessing pipeline consists of:
1. **Gensim simple_preprocess**: Tokenizes text, converts to lowercase, removes punctuation/numbers, and ignores short tokens (< 3 characters).
2. **Standard Stopword Removal**: Filters out common English function words using Gensim's STOPWORDS.
3. **Lemmatization (Nouns & Verbs)**: Uses NLTK's WordNetLemmatizer to reduce words to their root forms (e.g., "running" -> "run", "players" -> "player").
4. **Data-driven Frequency Filtering**: Rather than manually guessing and introducing human bias, we let Gensim's `filter_extremes` statistically filter out overly ubiquitous words in the next step.


In [2]:
import nltk
from nltk.stem import WordNetLemmatizer
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS

# Download WordNet data
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # 1. Tokenize and normalize
    tokens = simple_preprocess(text, deacc=True, min_len=3)
    # 2. Stopword removal and lemmatization (verb + noun)
    cleaned = [
        lemmatizer.lemmatize(lemmatizer.lemmatize(word, pos="v"), pos="n")
        for word in tokens
        if word not in STOPWORDS
    ]
    return cleaned

# Apply preprocessing to all documents
df["tokens"] = df["text"].apply(preprocess_text)

# Inspect before and after
print("Original Text Sample (first 200 chars):")
print(df.loc[0, "text"][:200])
print("\nPreprocessed Tokens Sample (first 25 tokens):")
print(df.loc[0, "tokens"][:25])


Original Text Sample (first 200 chars):
Ad sales boost Time Warner profit

Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (Â£600m) for the three months to December, from $639m year-earlier.

The firm, which is now one 

Preprocessed Tokens Sample (first 25 tokens):
['sale', 'boost', 'time', 'warner', 'profit', 'quarterly', 'profit', 'medium', 'giant', 'timewarner', 'jump', 'month', 'december', 'year', 'earlier', 'firm', 'biggest', 'investor', 'google', 'benefit', 'sale', 'high', 'speed', 'internet', 'connection']


## Part 3: Bag-of-Words Corpus & Statistical Dictionary Filtering

LDA requires documents in Bag-of-Words (BoW) format: a list of `(word_id, word_frequency)` pairs for each document.

To eliminate overly ubiquitous reporting words and noise in a purely data-driven way:
* **`no_below=5`**: Removes words appearing in fewer than 5 documents (rare typos, one-off names).
* **`no_above=0.5`**: Removes words appearing in more than 50% of all articles (statistically removes corpus-wide words like "say" at 87.4% and "year" at 65.4% without manual bias).
* We keep all vocabulary words that satisfy these statistical thresholds.


In [3]:
from gensim import corpora

# Create Gensim dictionary from tokenized documents
dictionary = corpora.Dictionary(df["tokens"])
print(f"Initial vocabulary size: {len(dictionary)}")

# Apply statistical filtering (no manual stopword bias)
dictionary.filter_extremes(no_below=5, no_above=0.5, keep_n=None)
print(f"Vocabulary size after statistical filtering: {len(dictionary)}")

# Convert each document into Bag-of-Words (BoW) format: (word_id, count)
corpus = [dictionary.doc2bow(tokens) for tokens in df["tokens"]]
print(f"Total documents in BoW corpus: {len(corpus)}")

# Display an example of the BoW representation for the first document
doc_0_bow = corpus[0][:10]
decoded_sample = [(dictionary[word_id], count) for word_id, count in doc_0_bow]
print("\nFirst 10 BoW tuples for Document 0 (ID, count):", doc_0_bow)
print("Decoded example:", decoded_sample)


Initial vocabulary size: 20680


Vocabulary size after statistical filtering: 6638
Total documents in BoW corpus: 2225

First 10 BoW tuples for Document 0 (ID, count): [(0, 2), (1, 1), (2, 1), (3, 2), (4, 1), (5, 1), (6, 7), (7, 1), (8, 1), (9, 1)]
Decoded example: [('account', 2), ('adjust', 1), ('advert', 1), ('advertise', 2), ('alexander', 1), ('analyst', 1), ('aol', 7), ('aside', 1), ('benefit', 1), ('better', 1)]


## Part 4: Train LDA Topic Models & Experiment with Topic Counts

We train Latent Dirichlet Allocation (LDA) models using Gensim's `models.LdaModel`.
To observe how the number of topics affects the learned representations, we experiment with three different topic counts:
* **$K = 3$ Topics**: Under-clustered, broad themes forced to merge.
* **$K = 5$ Topics**: Aligned with the 5 known BBC news categories (`business`, `entertainment`, `politics`, `sport`, `tech`).
* **$K = 10$ Topics**: Higher granularity, discovering more specific sub-themes within categories.

For a fair and reproducible comparison, we fix `passes=10` and `random_state=42`.


In [4]:
from gensim import models

RANDOM_STATE = 42

def train_lda(num_topics, passes=10):
    return models.LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=num_topics,
        passes=passes,
        random_state=RANDOM_STATE
    )

def print_model_topics(model, num_words=8):
    for topic_id, topic_str in model.print_topics(num_topics=model.num_topics, num_words=num_words):
        print(f"Topic {topic_id}: {topic_str}")

# Train models with 3, 5, and 10 topics
print("Training LDA with 3 topics...")
lda_3 = train_lda(3)

print("Training LDA with 5 topics...")
lda_5 = train_lda(5)

print("Training LDA with 10 topics...")
lda_10 = train_lda(10)
print("Training complete!\n")

print("=" * 60)
print("LDA Model: 3 Topics")
print("=" * 60)
print_model_topics(lda_3)

print("\n" + "=" * 60)
print("LDA Model: 5 Topics")
print("=" * 60)
print_model_topics(lda_5)

print("\n" + "=" * 60)
print("LDA Model: 10 Topics")
print("=" * 60)
print_model_topics(lda_10)


Training LDA with 3 topics...


Training LDA with 5 topics...


Training LDA with 10 topics...


Training complete!

LDA Model: 3 Topics
Topic 0: 0.024*"game" + 0.008*"market" + 0.008*"player" + 0.008*"new" + 0.007*"mobile" + 0.007*"technology" + 0.006*"dvd" + 0.006*"music"
Topic 1: 0.014*"people" + 0.010*"service" + 0.008*"phone" + 0.008*"user" + 0.008*"technology" + 0.007*"mobile" + 0.007*"net" + 0.006*"firm"
Topic 2: 0.005*"win" + 0.005*"time" + 0.005*"play" + 0.005*"world" + 0.004*"government" + 0.004*"new" + 0.004*"game" + 0.003*"come"

LDA Model: 5 Topics
Topic 0: 0.008*"government" + 0.006*"company" + 0.006*"minister" + 0.005*"new" + 0.005*"plan" + 0.005*"law" + 0.005*"election" + 0.005*"labour"
Topic 1: 0.016*"people" + 0.014*"service" + 0.009*"net" + 0.008*"user" + 0.008*"broadband" + 0.008*"search" + 0.007*"technology" + 0.006*"internet"
Topic 2: 0.008*"win" + 0.006*"play" + 0.006*"film" + 0.006*"time" + 0.005*"best" + 0.005*"game" + 0.005*"world" + 0.004*"new"
Topic 3: 0.022*"game" + 0.018*"mobile" + 0.012*"phone" + 0.011*"people" + 0.010*"music" + 0.010*"technology" + 

## Part 5: Visualize Topics with PyLDAvis

We use `pyLDAvis` to visually explore the learned topic distributions.
* **Left Panel (Intertopic Distance Map)**:
  * Each bubble corresponds to a topic projected onto a 2D space (PC1 and PC2).
  * **Bubble size**: Proportional to the topic's overall prevalence across the corpus.
  * **Distance / Overlap**: Distances approximate topic semantic similarity in 2D. Overlap suggests shared vocabulary or related themes.
* **Right Panel (Top Terms)**:
  * Shows prominent keywords for the selected topic.
  * **Blue bars**: Overall term frequency across the entire corpus.
  * **Red bars**: Estimated term frequency specific to the selected topic.
  * The relevance slider $\lambda$ adjusts ranking between high-frequency words ($\lambda=1$) and distinctive/exclusive words ($\lambda=0$).

We visualize the **5-topic model** with `sort_topics=False` so topic indices match `lda_5.print_topics()` directly.


In [5]:
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

# Prepare PyLDAvis visualization (sort_topics=False maintains consistent topic numbering)
vis_5 = gensimvis.prepare(lda_5, corpus, dictionary, sort_topics=False)

# Save to an standalone HTML file for convenient full-screen viewing
pyLDAvis.save_html(vis_5, "lda_vis_5_topics.html")
print("Visualization saved to lda_vis_5_topics.html")

# Display inline in notebook
pyLDAvis.display(vis_5)


Visualization saved to lda_vis_5_topics.html


## Part 6: Lab Questions & Discussion

Use the model outputs and the interactive PyLDAvis visualization above (or `lda_vis_5_topics.html`) to write your answers below.

---

### Question 1: Interpreting `lda_model.print_topics()`
> **Prompt:** When you use `lda_model.print_topics()` in Gensim, each topic is shown with words and numbers (e.g., `0.032*"food" + 0.020*"restaurant"`).  
> What do these numbers mean, and how can they be interpreted in the context of topic modeling?

*Your Answer:*
the numbers next to the word is the probability distribution of that word appearance in that specific topics, which if it is higher or closer to 1 it would be semantically closer to the central point of the topic

---

### Question 2: Interpreting the PyLDAvis Output
> **Prompt:** From the PyLDAvis output:
> 1. What does the size of each circle indicate?
> 2. What does the distance between circles tell you about the topics?
> 3. When you click on a topic, the right panel displays a list of words with bars of different colors and lengths. What do these bars represent, and what is the difference between the red and blue bars?

*Your Answer:*
1. size means how much of the topic contribute to the overall corpus. we talking about word in corpus not the document level since one document could belong to different topic
2. distance, assuming from the axes labeling 'PC1','PC2' which we can assume it used PCA to make the visual possible. distance between each circle means how much they are related or not - like a k-mean clustering
3. on each topic, after we clicked shows the blue bar and red bar, which blue responsible for distribution of that word to overall corpus and red is the distribution of that word to that specific topics.
---

### Question 3: Topic Overlap and Proximity
> **Prompt:** Do any topics overlap or appear close to each other in the visualization? What might this suggest about the similarity or distinctness of those topics?

*Your Answer:*
from `k=5` which means we directly using the same number of topics from the original topic provided by bbc, they don't significantly overlap by any mean. but there are some topic that are closer together. like the one we will be discussing on q5

---

### Question 4: Largest Bubble in PyLDAvis
> **Prompt:** Which topic appears to be the largest bubble in the visualization? What might that imply about your dataset?

*Your Answer:*
we have 2 largest bubble which is topic 1 and topic 3. From our observation it is political and sports respetively. and to assume after that observation, both of them provide a large list of word into our corpus - also some terms in the sports are very general so it could fit into multiple topics too

---

### Question 5: Comparison of Two Close Topics
> **Prompt:** Select two topics that appear close together in the visualization. Compare their top words — are there shared terms or themes? What does that tell you?

*Your Answer:*
there are 2 topics that are closely related to each other 2 and 5, 2 seems to be tech ones since it include many tech related companies in the corpus. but 5 mostly about tech in cybersecurity terms, like virus, malware, files, etc. and topic number 4 is further but still close to 2 and 5 being topic about gaming industry - mentioning game, mobile, console, etc.
